## ECS launch types, networking & scaling

The same task definition runs under two **launch types** — the difference is *who owns the host*. **EC2 launch type**: EC2 instances you (or an ASG) manage — you choose instance type and AMI, can SSH in, pay per instance-hour (idle costs money), and patch the fleet. **Fargate launch type**: AWS owns the host — invisible, no maintenance, billed per **task vCPU + memory per second**, no idle cost. Fargate uses discrete vCPU/memory pairs (0.25 vCPU / 0.5 GB up to 16 vCPU / 120 GB). The rule: **start with Fargate for simplicity; move to EC2 launch type + Savings Plans** once utilisation is steady and per-hour beats per-second.

**Networking**: almost always **`awsvpc`** mode, where each task gets its **own ENI, private IP, and security group** in your VPC — treat tasks like small EC2s on the network (it's required for Fargate). The legacy `bridge` and `host` modes are EC2-only. Attach the service to an **ALB/NLB target group** and ECS auto-registers tasks on launch, deregisters on stop.

**Scaling** uses **Application Auto Scaling** (the same engine as DynamoDB/Aurora): target tracking (hold CPU at 70%, or requests-per-target), step, and scheduled. For the **EC2 launch type** you must also scale the underlying instances — an **ECS Capacity Provider** backed by an ASG does that automatically. Fargate has no capacity to manage, which is much of its appeal.